In [ ]:
import scanpy as sc
import numpy as np
import flowkit as fk
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from src.preprocessing import read_flow
df_flow, df_list, df_wsp = read_flow(
    '11-Mar-2026 Treg .wsp', 'Treg FCS', 'T Cells')

In [ ]:
df_flow['tissue'].value_counts()

In [ ]:
df_flow.drop(columns=['FSC-A FSC - Area', 'FSC-H FSC - Height',
                      'FSC-W FSC - Width', 'SSC-A SSC - Area', 'SSC-H SSC - Height',
                      'SSC-W SSC - Width', 'AIM Dump', 'L/D', 'TIME Time Stamp',  '[AF color 1]-A [AF color 1] - Area', 'CD45', 'CD3'], inplace=True)

In [ ]:
df_flow_counts = df_flow.select_dtypes(include=[np.number])

In [ ]:
from src.preprocessing import pd_to_adata
adata = pd_to_adata(df_flow, df_flow_counts)

In [ ]:
adata_list = [
    adata[adata.obs['tissue'] == tissue].copy()[np.random.choice(
        adata[adata.obs['tissue'] == tissue].shape[0], 13987, replace=False
    )]
    for tissue in ['BLD', 'LLN', 'SPL', 'CLPD']
]

for i, tissue in enumerate(['BLD', 'LLN', 'SPL', 'CLPD']):
    adata_list[i].obs['tissue'] = tissue

adata = sc.AnnData.concatenate(*adata_list, index_unique=None)

In [ ]:
adata.X = np.arcsinh(adata.X / 150)

In [ ]:
sc.pp.scale(adata, max_value=5)

In [ ]:
sc.tl.pca(adata, svd_solver="arpack")
sc.pl.pca_variance_ratio(adata, log=True)

In [ ]:
sc.pl.pca_loadings(adata, components='1,2')

In [ ]:
import harmonypy as hm
harmony_out = hm.run_harmony(
    adata.obsm['X_pca'], adata.obs, 'sample_id', max_iter_harmony=10, theta=0)
adata.obsm['X_pca_harmony'] = harmony_out.Z_corr

In [ ]:
sc.pp.neighbors(adata)

In [ ]:
sc.tl.umap(adata)

In [ ]:
markers = list(adata.var_names)

In [ ]:
sc.pl.umap(adata, color=['group'], cmap='turbo')

In [ ]:
sc.tl.leiden(adata, resolution=0.5, flavor='leidenalg')

In [ ]:
sc.pl.umap(adata, color=['leiden'], cmap='turbo')

In [ ]:
sc.pl.umap(adata, color=['tissue'], cmap='turbo')

In [ ]:
sc.pl.umap(adata, color=['sample_id'], cmap='turbo')

In [ ]:
plt.rcParams.update({'font.size': 14})

sc.pl.umap(
    adata,
    color=markers,
    cmap='turbo',
    vmin=0,
    vmax=5,
)

In [ ]:
for tissue in list(adata.obs.tissue.unique()):
    adata_group = adata[adata.obs['tissue'] == tissue]

    sc.pl.umap(
        adata_group,
        color=['leiden'],
        title=f'{tissue} Leiden',
        cmap='turbo',
        show=False
    )

    ax = plt.gca()
    for cluster in adata_group.obs['leiden'].cat.categories:
        cluster_mask = adata_group.obs['leiden'] == cluster
        cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
        x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()
        ax.text(x, y, cluster, color='black', fontsize=10,
                weight='bold', ha='center', va='center')

    plt.show()

In [ ]:
sc.pl.dotplot(adata, markers, swap_axes=True, groupby='leiden',
              cmap='RdBu_r', dendrogram=True, vcenter=0, vmin=-4, vmax=4)

In [ ]:
marker_genes = {
    'CD4': ['CD4'],
    'CD8': ['CD8'],
    'TRM': ['CD103', 'CD69'],
    'Memory': ['CD45RA', 'CCR7']
}

In [ ]:
sc.pl.dotplot(adata, marker_genes, swap_axes=True, groupby='leiden',
              cmap='RdBu_r', dendrogram=True, vcenter=0, vmin=-4, vmax=4)

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden", method="t-test")

result = adata.uns["rank_genes_groups"]
groups = result["names"].dtype.names

celltype = {'celltype': []}
cluster_to_genes = {}
for group in groups:
    top_genes = result["names"][group][:3]
    cluster_to_genes[group] = f"{':'.join(top_genes)} ({group})"

celltype['celltype'] = [cluster_to_genes[leiden]
                        for leiden in adata.obs['leiden']]

In [ ]:
cell_type_series = pd.Series(celltype['celltype'])
unique_values = cell_type_series.unique()
print(unique_values)

In [ ]:
cluster_annotation = {'celltype': []}
cluster_to_genes = {
    '0': 'HLA-DR+ CD45RA+CCR7+ (0)',
    '2': 'HLA-DR+ CD45RA-CCR7- (2)',
    '7': 'CD4+ TEMRA (7)',
    '3': 'CD8+ EM CD103+CD69+ (3)',
    '6': 'CD8+ TEMRA (6)',
    '5': 'CD4+ TCM CD69+ (5)',
    '4': 'CD4+ TCM (4)',
    '1': 'CD4+ EM (1)',
    '8': 'CD4+ EM CD103+CD69+ (8)'

}
cluster_annotation['celltype'] = [cluster_to_genes[leiden]
                                  for leiden in adata.obs['leiden']]
adata.obs["leiden_annotation"] = cluster_annotation['celltype']

print(adata.obs[["leiden", "leiden_annotation"]].head())

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

sc.pl.umap(
    adata,
    color=['leiden_annotation'],
    cmap='turbo',
    title='CyTOF Annotated Leiden Clusters',
    show=False,
)

ax = plt.gca()
for cluster in adata.obs['leiden'].cat.categories:
    cluster_mask = adata.obs['leiden'] == cluster
    cluster_coords = adata.obsm['X_umap'][cluster_mask]
    x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()
    ax.text(x, y, cluster, color='black', fontsize=10,
            weight='bold', ha='center', va='center')

plt.show()

In [ ]:
for tissue in list(adata.obs.tissue.unique()):
    adata_group = adata[adata.obs['tissue'] == tissue]

    sc.pl.umap(
        adata_group,
        color=['leiden_annotation'],
        title=f'{tissue} Clusters',
        cmap='turbo',
        show=False
    )

    ax = plt.gca()
    for cluster in adata_group.obs['leiden'].cat.categories:
        cluster_mask = adata_group.obs['leiden'] == cluster
        cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
        x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()
        ax.text(x, y, cluster, color='black', fontsize=10,
                weight='bold', ha='center', va='center')

    plt.show()

In [ ]:
for group in list(adata.obs.group.unique()):
    adata_group = adata[adata.obs['group'] == group]

    sc.pl.umap(
        adata_group,
        color=['leiden_annotation'],
        title=f'{group} Clusters',
        cmap='turbo',
        show=False
    )

    ax = plt.gca()
    for cluster in adata_group.obs['leiden'].cat.categories:
        cluster_mask = adata_group.obs['leiden'] == cluster
        cluster_coords = adata_group.obsm['X_umap'][cluster_mask]
        x, y = cluster_coords[:, 0].mean(), cluster_coords[:, 1].mean()
        ax.text(x, y, cluster, color='black', fontsize=10,
                weight='bold', ha='center', va='center')

    plt.show()